In [ ]:
%%capture
!tar -xvf data.tar

In [ ]:
!pip install --upgrade huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from tests.adapters import run_parse_mmlu_response


In [ ]:
root_directory = 'data/test'
dfs = []  # Initialize an empty list to store individual DataFrames
combined_df = None
col_names = ["question", "option_a", "option_b", "option_c", "option_d", "answer"]

for root, dirs, files in os.walk(root_directory):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            try:
                df = pd.read_csv(file_path, names=col_names, header=None)
                df['subject'] = file.removesuffix("_test.csv").replace('_', ' ')
                dfs.append(df)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

if dfs:  # Check if any DataFrames were loaded
    combined_df = pd.concat(dfs, axis='rows', ignore_index=True)
    print("Combined DataFrame created successfully.")
    # You can now work with combined_df
else:
    print("No CSV files found or processed.")

In [ ]:
combined_df

In [ ]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='meta-llama/Llama-3.1-8B')

# Read fine tuned model from local
# llm = LLM(model='sft_model/')

In [ ]:
df_idx = 100
question = combined_df.iloc[df_idx]['question']
subject = combined_df.iloc[df_idx]['subject']
option_a = combined_df.iloc[df_idx]['option_a']
option_b = combined_df.iloc[df_idx]['option_b']
option_c = combined_df.iloc[df_idx]['option_c']
option_d = combined_df.iloc[df_idx]['option_d']
answer = combined_df.iloc[df_idx]['answer']

instruction = f"""
Answer the following multiple choice question about {subject}. Respond with a single
sentence of the form "The correct answer is _", filling the blank with the letter
corresponding to the correct answer (i.e., A, B, C or D).
Question: {question}
A. {option_a}
B. {option_b}
C. {option_c}
D. {option_d}
Answer:
"""

prompt = f"""
# Instruction
Below is a list of conversations between a human and an AI assistant (you).
Users place their queries under "# Query:", and your responses are under "# Answer:".
You are a helpful, respectful, and honest assistant.
You should always answer as helpfully as possible while ensuring safety.
Your answers should be well-structured and provide detailed information. They should also
have an engaging tone. Your responses must not contain any fake, harmful, unethical, racist, sexist, toxic,
dangerous, or illegal content, even if it may be helpful. Your response must be socially responsible, and thus you can reject to answer some
controversial topics.
# Query:
```{instruction}```
# Answer:
```
"""

In [ ]:
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["```"], include_stop_str_in_output=True
)

outputs = llm.generate([prompt], sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")
    print(f"Extraced answer: {run_parse_mmlu_response({}, generated_text)}")


In [ ]:
from typing import Literal
def evaluate_vllm(
  vllm_model: LLM,
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) -> dict[Literal["correct", "incorrect", "failed_to_extract"], int] :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)

  results = {"correct": 0, "incorrect": 0, "failed_to_extract": 0}

  for index, output in enumerate(outputs):
    generated_text = output.outputs[0].text
    extracted_answer = run_parse_mmlu_response({}, generated_text)
    if extracted_answer is None:
        results['failed_to_extract'] += 1
    elif extracted_answer == answers[index]:
        results['correct'] += 1
    else:
        results['incorrect'] += 1

  return results

In [ ]:
total_correct = 0
total_incorrect = 0
total_failed_to_extract = 0
prompts = []
answers = []

for index, row in tqdm(combined_df.iterrows()):
    question = row['question']
    subject = row['subject']
    option_a = row['option_a']
    option_b = row['option_b']
    option_c = row['option_c']
    option_d = row['option_d']
    answer = row['answer']

    instruction = f"""
    Answer the following multiple choice question about {subject}. Respond with a single
    sentence of the form "The correct answer is _", filling the blank with the letter
    corresponding to the correct answer (i.e., A, B, C or D).
    Question: {question}
    A. {option_a}
    B. {option_b}
    C. {option_c}
    D. {option_d}
    Answer:
    """
    prompt = f"""
    # Instruction
    Below is a list of conversations between a human and an AI assistant (you).
    Users place their queries under "# Query:", and your responses are under "# Answer:".
    You are a helpful, respectful, and honest assistant.
    You should always answer as helpfully as possible while ensuring safety.
    Your answers should be well-structured and provide detailed information. They should also
    have an engaging tone. Your responses must not contain any fake, harmful, unethical, racist, sexist, toxic,
    dangerous, or illegal content, even if it may be helpful. Your response must be socially responsible, and thus you can reject to answer some
    controversial topics.
    # Query:
    ```{instruction}```
    # Answer:
    ```
    """

    prompts.append(prompt)
    answers.append(answer)

    if (index + 1) % 240 == 0:
        results = evaluate_vllm(llm, prompts, answers, sampling_params)

        # Add batch results to totals
        total_correct += results['correct']
        total_incorrect += results['incorrect']
        total_failed_to_extract += results['failed_to_extract']

        print(f"Total correct so far: {total_correct}")
        print(f"Total incorrect so far: {total_incorrect}")
        print(f"Total total_failed_to_extract so far: {total_failed_to_extract}")
        print(f"Pass rate: {total_correct / (total_correct + total_incorrect + total_failed_to_extract)}")
        prompts = []
        answers = []

print(f"Total correct reward: {total_correct}")
print(f"Total incorrect reward: {total_incorrect}")
print(f"Total total_failed_to_extract: {total_failed_to_extract}")
print(f"Pass rate: {total_correct / (total_correct + total_incorrect + total_failed_to_extract)}")

In [ ]:
# Total correct reward: 3819
# Total incorrect reward: 5734
# Total total_failed_to_extract: 4367
# 27% pass rate